# 🧠 EX53: Training Settings

| Param | Default | Effect |
|-------|---------|--------|
| `epochs` | 100 | Training rounds |
| `batch` | 16 | Samples per gradient step |
| `imgsz` | 640 | Input resolution (px) |
| `lr0` | 0.01 | Initial LR |
| `optimizer` | `AdamW` | Gradient descent algorithm |
| `amp` | True | Mixed precision FP16 — halves VRAM |

### AdamW vs SGD
**SGD+momentum:** $v_{t+1} = \mu v_t - \eta \nabla L$, $\theta \mathrel{+}= v_{t+1}$

**AdamW:** bias-corrected adaptive LR per parameter + decoupled weight decay.
AdamW converges faster; SGD+cosine often peaks higher on large-scale data.

## ⚠️ Safety Warning
Verify `data.yaml` before `model.train()`.

## 🔗 Links
- [[YOLO_Learning_Plan]] | [[learning_journal]]


In [ ]:
# Back up or checkpoint this section of code before starting to modify the large file.
import gc, torch
import pandas as pd
import matplotlib.pyplot as plt
from solution import train_custom_model
%matplotlib inline

device = "0" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory // 1024**3
    print(f"[INFO] Device: GPU:{device} | VRAM: {vram} GB")
else:
    print("[INFO] Device: cpu")

print("\n--- AUDIT & INSPECTION START ---")
print("Training 3 epochs on coco8 to audit loss output...")
results = train_custom_model(model_path="yolo11n.pt", data_yaml="coco8.yaml",
                              epochs=3, batch=8, imgsz=320, device=device)

results_csv = results.save_dir / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    print(f"\nCSV columns: {list(df.columns)}")
    print(df.tail(3).to_string(index=False))
    loss_cols = [c for c in df.columns if "loss" in c.lower()]
    fig, axes = plt.subplots(1, len(loss_cols), figsize=(4*len(loss_cols), 4))
    if len(loss_cols)==1: axes=[axes]
    for ax,col in zip(axes, loss_cols):
        ax.plot(df["epoch"], df[col], marker="o", linewidth=2)
        ax.set_title(col); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.grid(True, alpha=0.3)
    plt.suptitle("Training Loss Curves (3-epoch audit)")
    plt.tight_layout(); plt.show()
else:
    print("results.csv not found.")
print("--- AUDIT & INSPECTION END ---")

gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
